In [5]:
import polars as pl
pl.Config.set_tbl_rows(-1)       # -1 means "show all rows"
pl.Config.set_tbl_cols(-1)       # -1 means "show all columns"
pl.Config.set_fmt_str_lengths(100)

polars.config.Config

In [19]:
import polars as pl

# 1. Scan the files
# We set 'ignore_errors=True' to handle lines that are physically broken 
# (e.g., lines with too many commas)
lazy_df = pl.scan_csv(
    "data/*.csv",
    infer_schema_length=10000,
    ignore_errors=True 
)

# 2. The "Skip Row" Logic

# 3. Stream to your final file
lazy_df.sink_parquet("IDS2018_cleaned_v2.parquet")

In [20]:
# look for null values in the final parquet file
final_df = pl.read_parquet("IDS2018_cleaned_v2.parquet")
null_counts = final_df.null_count()
print(f"Null value counts per column:{null_counts}")

Null value counts per column:shape: (1, 80)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┬─────┐
│ Dst ┆ Pro ┆ Tim ┆ Flo ┆ Tot ┆ Tot ┆ Tot ┆ Tot ┆ Fwd ┆ Fwd ┆ Fwd ┆ Fwd ┆ Bwd ┆ Bwd ┆ Bwd ┆ Bwd ┆ Flo ┆ Flo ┆ Flo ┆ Flo ┆ Flo ┆ Flo ┆ Fwd ┆ Fwd ┆ Fwd ┆ Fwd ┆ Fwd ┆ Bwd ┆ Bwd ┆ Bwd ┆ Bwd ┆ Bwd ┆ Fwd ┆ Bwd ┆ Fwd ┆ Bwd ┆ Fwd ┆ Bwd ┆ Fwd ┆ Bwd ┆ Pkt ┆ Pkt ┆ Pkt ┆ Pkt ┆ Pkt ┆ FIN ┆ SYN ┆ RST ┆ PSH ┆ ACK ┆ URG ┆ CWE ┆ ECE ┆ Dow ┆ Pkt ┆ Fwd ┆ Bwd ┆ Fwd ┆ Fwd ┆ Fwd ┆ Bwd ┆ Bwd ┆ Bwd ┆ Sub ┆ Sub ┆ Sub ┆ Sub ┆ Ini ┆ Ini ┆ Fwd ┆ Fwd ┆ Act ┆ Act ┆ Act ┆ Act ┆ Idl ┆ Idl ┆ Idl ┆ Idl 

In [1]:
import polars as pl

# 1. Scan lazily
lazy_df = pl.scan_parquet("IDS2018_cleaned_v2.parquet")
print("Plan loaded. Analyzing label distribution...")

# 2. Get counts (Using .len() to avoid the DeprecationWarning)
label_counts = lazy_df.select("Label").group_by("Label").len().collect()
print(label_counts)

train_frames = []
test_frames = []
val_frames = []

# 3. Stratified Split Logic
for label in label_counts["Label"]:
    # Filter for the class (Lazy)
    shuffled_lazy = lazy_df.filter(pl.col("Label") == label)
    
    # We SHUFFLE the class lazily first. 
    # This ensures that when we slice it, we get a random distribution.
    
    
    # Get the total count for this label to calculate slice offsets
    total_count = label_counts.filter(pl.col("Label") == label)["len"][0]
    
    # Calculate sizes for 70/20/10 split
    train_size = int(total_count * 0.7)
    test_size = int(total_count * 0.2)
    # Val size is the remainder
    
    # Use .slice(offset, length) - This is LAZY and RAM-friendly
    # Slice 1: Start at 0, take 70%
    train_frames.append(shuffled_lazy.slice(0, train_size))
    
    # Slice 2: Start after train, take 20%
    test_frames.append(shuffled_lazy.slice(train_size, test_size))
    
    # Slice 3: Start after test, take the rest
    val_frames.append(shuffled_lazy.slice(train_size + test_size, None))
    
    print(f"Added lazy slices for: {label}")

# 4. The SINK (The only part that actually processes the data)
print("Sinking to disk... This avoids loading the 2GB into RAM.")

pl.concat(train_frames).sink_parquet("train_data.parquet")
pl.concat(test_frames).sink_parquet("test_data.parquet")
pl.concat(val_frames).sink_parquet("val_data.parquet")

print("Stratified split complete! 3 files generated: train, test, val.")

Plan loaded. Analyzing label distribution...
shape: (15, 2)
┌──────────────────────────┬─────────┐
│ Label                    ┆ len     │
│ ---                      ┆ ---     │
│ str                      ┆ u32     │
╞══════════════════════════╪═════════╡
│ Infilteration            ┆ 161934  │
│ SSH-Bruteforce           ┆ 187589  │
│ DoS attacks-GoldenEye    ┆ 41508   │
│ DoS attacks-Hulk         ┆ 461912  │
│ DoS attacks-SlowHTTPTest ┆ 139890  │
│ …                        ┆ …       │
│ FTP-BruteForce           ┆ 193360  │
│ DDOS attack-LOIC-UDP     ┆ 1730    │
│ Label                    ┆ 59      │
│ DDOS attack-HOIC         ┆ 686012  │
│ Benign                   ┆ 6112151 │
└──────────────────────────┴─────────┘
Added lazy slices for: Infilteration
Added lazy slices for: SSH-Bruteforce
Added lazy slices for: DoS attacks-GoldenEye
Added lazy slices for: DoS attacks-Hulk
Added lazy slices for: DoS attacks-SlowHTTPTest
Added lazy slices for: Bot
Added lazy slices for: Brute Force -Web
Ad

In [2]:
import polars as pl

# 1. Scan the raw data
lf = pl.scan_parquet("train_data.parquet")

# 2. Define the cleaning plan
exclude = ['Timestamp', 'Label']
numeric_cols = [col for col in lf.columns if col not in exclude]

# 3. Apply cleaning AND the sink in one go
(
    lf.with_columns([
        pl.col(numeric_cols).cast(pl.Float64, strict=False)
    ])
    .sink_parquet("train_data_str_handled.parquet")
)

print("✅ Cleaned without any manual batching errors!")

C:\Users\akhiz\AppData\Local\Temp\ipykernel_14560\387897372.py:8: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  numeric_cols = [col for col in lf.columns if col not in exclude]


✅ Cleaned without any manual batching errors!


In [3]:
import polars as pl

# 1. Load the plan
lf = pl.scan_parquet("train_data_str_handled.parquet")

# 2. Identify target columns
exclude = ['Timestamp', 'Label']
numeric_cols = [col for col in lf.columns if col not in exclude]

# 3. FORCE CASTING: This is the critical step
# We cast to Float64 and use 'strict=False' to turn "Infinity" or text into Null
lf_numeric = lf.with_columns([
    pl.col(numeric_cols)
])

# 4. Create the Analysis Plan
# We use .len() instead of .count()
analysis_plan = lf_numeric.select([
    pl.col(numeric_cols).n_unique().name.map(lambda c: f"{c}_unique"),
    pl.col(numeric_cols).null_count().name.map(lambda c: f"{c}_missing"),
    (pl.col(numeric_cols) == 0).sum().name.map(lambda c: f"{c}_zeros"),
    pl.col(numeric_cols).std().name.map(lambda c: f"{c}_std"),
    pl.col(numeric_cols).mean().name.map(lambda c: f"{c}_mean"),
    pl.len().alias("total_rows")
])

# 5. Sink to CSV
stats_output = "feature_stats.csv"
analysis_plan.sink_csv(stats_output)

print(f"✅ Statistics saved to {stats_output} - String errors bypassed!")

# 5. Now we read the TINY 1-row CSV back for processing
# (This file is only a few KB, so it's 100% safe)
stats_df = pl.read_csv(stats_output)
total_rows = stats_df["total_rows"][0]

# 6. Reformat for your report (Simple Python logic on a tiny DF) 
# handle division by 0 error for missing values

report_data = []
for col in numeric_cols:
    report_data.append({
        "Feature": col,
        "Unique_Values": stats_df[f"{col}_unique"][0],
        "Missing_%": (stats_df[f"{col}_missing"][0] / total_rows) * 100,
        "Zero_%": (stats_df[f"{col}_zeros"][0] / total_rows) * 100,
        "Std_Dev": stats_df[f"{col}_std"][0],
        "Mean": stats_df[f"{col}_mean"][0]
    })

# Turn this small list into a Final Analysis DataFrame
final_report = pl.DataFrame(report_data)
print(final_report)


C:\Users\akhiz\AppData\Local\Temp\ipykernel_14560\3720311970.py:8: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  numeric_cols = [col for col in lf.columns if col not in exclude]


✅ Statistics saved to feature_stats.csv - String errors bypassed!
shape: (78, 6)
┌───────────────┬───────────────┬───────────┬───────────┬──────────────┬──────────────┐
│ Feature       ┆ Unique_Values ┆ Missing_% ┆ Zero_%    ┆ Std_Dev      ┆ Mean         │
│ ---           ┆ ---           ┆ ---       ┆ ---       ┆ ---          ┆ ---          │
│ str           ┆ i64           ┆ f64       ┆ f64       ┆ f64          ┆ f64          │
╞═══════════════╪═══════════════╪═══════════╪═══════════╪══════════════╪══════════════╡
│ Dst Port      ┆ 62628         ┆ 0.000707  ┆ 1.039667  ┆ 21559.111333 ┆ 12674.840954 │
│ Protocol      ┆ 4             ┆ 0.000707  ┆ 1.039667  ┆ 4.405149     ┆ 8.049618     │
│ Flow Duration ┆ 1948085       ┆ 0.000707  ┆ 0.390742  ┆ 8.2542e8     ┆ 9.6086e6     │
│ Tot Fwd Pkts  ┆ 2866          ┆ 0.000707  ┆ 0.0       ┆ 1769.577453  ┆ 30.305363    │
│ Tot Bwd Pkts  ┆ 2286          ┆ 0.000707  ┆ 28.107366 ┆ 152.311014   ┆ 5.853412     │
│ …             ┆ …             ┆ …    

In [6]:
print(final_report)

shape: (78, 6)
┌───────────────────┬───────────────┬───────────┬───────────┬───────────────┬───────────────┐
│ Feature           ┆ Unique_Values ┆ Missing_% ┆ Zero_%    ┆ Std_Dev       ┆ Mean          │
│ ---               ┆ ---           ┆ ---       ┆ ---       ┆ ---           ┆ ---           │
│ str               ┆ i64           ┆ f64       ┆ f64       ┆ f64           ┆ f64           │
╞═══════════════════╪═══════════════╪═══════════╪═══════════╪═══════════════╪═══════════════╡
│ Dst Port          ┆ 62628         ┆ 0.000707  ┆ 1.039667  ┆ 21559.111333  ┆ 12674.840954  │
│ Protocol          ┆ 4             ┆ 0.000707  ┆ 1.039667  ┆ 4.405149      ┆ 8.049618      │
│ Flow Duration     ┆ 1948085       ┆ 0.000707  ┆ 0.390742  ┆ 8.2542e8      ┆ 9.6086e6      │
│ Tot Fwd Pkts      ┆ 2866          ┆ 0.000707  ┆ 0.0       ┆ 1769.577453   ┆ 30.305363     │
│ Tot Bwd Pkts      ┆ 2286          ┆ 0.000707  ┆ 28.107366 ┆ 152.311014    ┆ 5.853412      │
│ TotLen Fwd Pkts   ┆ 13335         ┆ 0.00070

In [8]:
import polars as pl
import os
import shutil

input_path = "train_data_str_handled.parquet"
output_path = "train_data_deduplicated.parquet"
bucket_dir = "dup_check_buckets"
os.makedirs(bucket_dir, exist_ok=True)

print("🚀 Step 1: Partitioning data using Row-Level Hashing...")
lf = pl.scan_parquet(input_path)

# THE FIX: Use pl.struct(pl.all()).hash() to get ONE hash per row
for i in range(10):
    (
        lf.filter(
            pl.struct(pl.all()).hash(seed=42).mod(10) == i
        )
        .sink_parquet(f"{bucket_dir}/bucket_{i}.parquet")
    )
    print(f"✅ Bucket {i} written to disk.")

    
print("\n🧹 Step 2: Removing duplicates within buckets...")
for i in range(10):
    bucket_file = f"{bucket_dir}/bucket_{i}.parquet"
    if not os.path.exists(bucket_file): continue
    
    # Read the bucket and apply .unique()
    # Since identical rows are in the same bucket, this catches all duplicates
    (
        pl.read_parquet(bucket_file)
        .unique() 
        .write_parquet(f"{bucket_dir}/cleaned_bucket_{i}.parquet")
    )
    
    os.remove(bucket_file)
    print(f"✨ Bucket {i} cleaned.")

# 3. Final Merge
print("\n🔗 Step 3: Merging buckets into final gold dataset...")
pl.scan_parquet(f"{bucket_dir}/cleaned_bucket_*.parquet").sink_parquet(output_path)

# 4. Final Verification
initial_rows = pl.scan_parquet(input_path).select(pl.len()).collect().item()
final_rows = pl.scan_parquet(output_path).select(pl.len()).collect().item()
removed = initial_rows - final_rows
retention = (final_rows / initial_rows) * 100

print("\n" + "="*80)
print("✅ GLOBAL DEDUPLICATION COMPLETE")
print("="*80)
print(f"📊 Initial rows:  {initial_rows:,}")
print(f"📊 Final rows:    {final_rows:,}")
print(f"📉 Rows removed:  {removed:,}")
print(f"📈 Retention:     {retention:.2f}%")

# Cleanup
shutil.rmtree(bucket_dir)


🚀 Step 1: Partitioning data using Row-Level Hashing...
✅ Bucket 0 written to disk.
✅ Bucket 1 written to disk.
✅ Bucket 2 written to disk.
✅ Bucket 3 written to disk.
✅ Bucket 4 written to disk.
✅ Bucket 5 written to disk.
✅ Bucket 6 written to disk.
✅ Bucket 7 written to disk.
✅ Bucket 8 written to disk.
✅ Bucket 9 written to disk.

🧹 Step 2: Removing duplicates within buckets...
✨ Bucket 0 cleaned.
✨ Bucket 1 cleaned.
✨ Bucket 2 cleaned.
✨ Bucket 3 cleaned.
✨ Bucket 4 cleaned.
✨ Bucket 5 cleaned.
✨ Bucket 6 cleaned.
✨ Bucket 7 cleaned.
✨ Bucket 8 cleaned.
✨ Bucket 9 cleaned.

🔗 Step 3: Merging buckets into final gold dataset...

✅ GLOBAL DEDUPLICATION COMPLETE
📊 Initial rows:  5,798,971
📊 Final rows:    5,529,768
📉 Rows removed:  269,203
📈 Retention:     95.36%


In [9]:
import polars as pl

# 1. SETUP & SCAN
input_path = "train_data_deduplicated.parquet"
output_path = "train_data_cleaned_invalid_inputs.parquet"

print("=" * 80)
print("🧹 POLARS CLEANING - INVALID FLOW REMOVAL")
print("=" * 80)

# Scan the data (Lazy - no RAM used yet)
lf = pl.scan_parquet(input_path)

# Initial State
initial_rows = lf.select(pl.len()).collect().item()
print(f"\n📊 Initial Dataset:")
print(f"    Total rows: {initial_rows:,}")
print(f"    Total columns: {len(lf.columns)}")

# =============================================================
# STEP 1 & 2: REMOVE DUPLICATES & INVALID FLOWS
# =============================================================
# We handle the 'Invalid Flow' check: (~(FwdPkts=0 AND BwdPkts=0 AND Duration=0))
# .fill_null(0) ensures we don't accidentally drop rows with valid Nulls
cleaned_lf = (
    lf
    .unique()  # Deduplication (redundant if already done, but safe)
    .filter(
        ~(
            (pl.col("Tot Fwd Pkts").fill_null(0) == 0) & 
            (pl.col("Tot Bwd Pkts").fill_null(0) == 0) & 
            (pl.col("Flow Duration").fill_null(0) == 0)
        )
    )
    # STEP 3: Remove completely empty/corrupt lines (dropna how='all')
    .filter(~pl.all_horizontal(pl.all().is_null()))
)

# =============================================================
# STEP 4: EXECUTION & REPORTING
# =============================================================
# We 'sink' the data to run the plan without crashing the kernel
print(f"\n🔍 Processing filters and saving to disk...")
cleaned_lf.sink_parquet(output_path)

# Scan the newly created file for final stats
final_lf = pl.scan_parquet(output_path)
final_rows = final_lf.select(pl.len()).collect().item()
removed = initial_rows - final_rows
retention = (final_rows / initial_rows) * 100

print("\n" + "=" * 80)
print("✅ CLEANING COMPLETE")
print("=" * 80)
print(f"📊 Final Dataset:")
print(f"    Total rows: {final_rows:,}")
print(f"    Rows removed: {removed:,}")
print(f"    Retention rate: {retention:.2f}%")

# =============================================================
# STEP 5: LABEL DISTRIBUTION
# =============================================================
if "Label" in final_lf.columns:
    print(f"\n🎯 Label Distribution After Cleaning:")
    # We group by label, count, and calculate percentage
    dist = (
        final_lf.group_by("Label")
        .len()
        .with_columns(
            (pl.col("len") / final_rows * 100).round(2).alias("%")
        )
        .sort("len", descending=True)
        .collect()
    )
    
    # Configure Polars to show all labels
    with pl.Config(tbl_rows=-1):
        print(dist)

print("\n" + "=" * 80)
print(f"💾 Cleaned dataset stored in '{output_path}'")
print("=" * 80)

🧹 POLARS CLEANING - INVALID FLOW REMOVAL

📊 Initial Dataset:
    Total rows: 5,529,768
    Total columns: 80

🔍 Processing filters and saving to disk...


C:\Users\akhiz\AppData\Local\Temp\ipykernel_14560\205153360.py:18: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  print(f"    Total columns: {len(lf.columns)}")



✅ CLEANING COMPLETE
📊 Final Dataset:
    Total rows: 5,529,767
    Rows removed: 1
    Retention rate: 100.00%

🎯 Label Distribution After Cleaning:


C:\Users\akhiz\AppData\Local\Temp\ipykernel_14560\205153360.py:63: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  if "Label" in final_lf.columns:


shape: (14, 3)
┌──────────────────────────┬─────────┬───────┐
│ Label                    ┆ len     ┆ %     │
│ ---                      ┆ ---     ┆ ---   │
│ str                      ┆ u32     ┆ f64   │
╞══════════════════════════╪═════════╪═══════╡
│ Benign                   ┆ 4270487 ┆ 77.23 │
│ DDOS attack-HOIC         ┆ 473210  ┆ 8.56  │
│ DoS attacks-Hulk         ┆ 302033  ┆ 5.46  │
│ Bot                      ┆ 198386  ┆ 3.59  │
│ Infilteration            ┆ 113328  ┆ 2.05  │
│ SSH-Bruteforce           ┆ 87471   ┆ 1.58  │
│ DoS attacks-GoldenEye    ┆ 29038   ┆ 0.53  │
│ FTP-BruteForce           ┆ 27496   ┆ 0.5   │
│ DoS attacks-SlowHTTPTest ┆ 19462   ┆ 0.35  │
│ DoS attacks-Slowloris    ┆ 6997    ┆ 0.13  │
│ DDOS attack-LOIC-UDP     ┆ 1211    ┆ 0.02  │
│ Brute Force -Web         ┆ 427     ┆ 0.01  │
│ Brute Force -XSS         ┆ 161     ┆ 0.0   │
│ SQL Injection            ┆ 60      ┆ 0.0   │
└──────────────────────────┴─────────┴───────┘

💾 Cleaned dataset stored in 'train_data_clea

In [11]:
import polars as pl

# --- CONFIGURATION ---
input_path = "train_data_cleaned_invalid_inputs.parquet" 
output_path = "train_data_optimized.parquet"

print("=" * 80)
print("POLARS: STREAMING FEATURE SELECTION & OPTIMIZATION")
print("=" * 80)

# 1. Scan the data to get available columns (Lazy, no RAM used)
lf = pl.scan_parquet(input_path)
available_columns = lf.columns

# =============================================================
# STEP 1: KEEP ONLY IMPORTANT FEATURES
# =============================================================
important_features = [
    'Timestamp', 'Src IP', 'Dst IP', 'Src Port', 'Dst Port', 'Protocol',
    'Flow Duration', 'Tot Fwd Pkts', 'Tot Bwd Pkts', 'Flow Byts/s', 'Flow Pkts/s',
    'Fwd Pkts/s', 'Bwd Pkts/s', 'Pkt Len Mean', 'Pkt Len Var', 'Pkt Len Std',
    'Fwd Pkt Len Mean', 'Bwd Pkt Len Mean', 'Fwd Pkt Len Std', 'Bwd Pkt Len Std',
    'Fwd Seg Size Avg', 'Bwd Seg Size Avg', 'Init Fwd Win Byts', 'Init Bwd Win Byts',
    'Active Mean', 'Idle Mean', 'Flow IAT Mean', 'Flow IAT Std',
    'Fwd IAT Mean', 'Bwd IAT Mean', 'Down/Up Ratio', 'Label'
]

# Ensure we only try to keep columns that actually exist in the file
keep_cols = [c for c in important_features if c in available_columns]
print(f"Selecting {len(keep_cols)} key features (Discarding the rest)")

# Identify categorical candidates dynamically
cat_candidates = [c for c in ['Protocol', 'Label'] if c in keep_cols]

# =============================================================
# STEP 2: BUILD THE OPTIMIZATION PLAN
# =============================================================
# THE FIX: Chain with_columns sequentially to avoid parallel assignment conflicts
optimized_lf = (
    lf.select(keep_cols)
    # Phase A: Downcast all numeric columns
    .with_columns([
        pl.col(pl.Float64).cast(pl.Float32),
        pl.col(pl.Int64).cast(pl.Int32),
    ])
    # Phase B: Convert specific columns to Categorical (Overwrites Phase A if needed)
    .with_columns([
        pl.col(cat_candidates).cast(pl.String).cast(pl.Categorical),
    ])
)

# Optional: Handle Timestamp safely if it exists
if 'Timestamp' in keep_cols:
    optimized_lf = optimized_lf.with_columns([
        pl.col('Timestamp').str.to_datetime(strict=False)
    ])

# =============================================================
# STEP 3: EXECUTE & SINK
# =============================================================
print("Processing and saving optimized dataset...")
optimized_lf.sink_parquet(output_path)

# =============================================================
# STEP 4: FINAL MEMORY REPORT
# =============================================================
df_final = pl.read_parquet(output_path)
mem_mb = df_final.estimated_size() / (1024 ** 2)

print("\n" + "=" * 80)
print("OPTIMIZATION COMPLETE")
print("=" * 80)
print(f"Final Shape: {df_final.shape}")
print(f"RAM footprint in Polars: {mem_mb:.2f} MB")

POLARS: STREAMING FEATURE SELECTION & OPTIMIZATION
Selecting 29 key features (Discarding the rest)
Processing and saving optimized dataset...


C:\Users\akhiz\AppData\Local\Temp\ipykernel_14560\3657797027.py:13: PerformanceWarning: Determining the column names of a LazyFrame requires resolving its schema, which is a potentially expensive operation. Use `LazyFrame.collect_schema().names()` to get the column names without this warning.
  available_columns = lf.columns



OPTIMIZATION COMPLETE
Final Shape: (5529767, 29)
RAM footprint in Polars: 632.83 MB


In [2]:
import polars as pl

print("Starting minimal feature engineering...")
# Assuming you loaded your data eagerly for notebook exploration:
df = pl.read_parquet("train_data_optimized.parquet")



Starting minimal feature engineering...


In [3]:
# ===== CELL 3: CLUSTER 1 - Traffic Balance (4 features) =====
"""
PURPOSE: Detect asymmetric communication patterns
- DDoS floods show extreme imbalance (many packets one way)
- Port scans have minimal responses
- Normal traffic is usually more balanced
"""
# We define total_pkts as a Polars expression for reuse
total_pkts_expr = pl.col('Tot Fwd Pkts') + pl.col('Tot Bwd Pkts')

df = df.with_columns([
    (pl.col('Tot Fwd Pkts') / (pl.col('Tot Bwd Pkts') + 1)).alias('fwd_bwd_pkt_ratio'),
    
    # min_horizontal replaces np.minimum
    pl.min_horizontal(
        pl.col('Tot Fwd Pkts') / (total_pkts_expr + 1),
        pl.col('Tot Bwd Pkts') / (total_pkts_expr + 1)
    ).alias('traffic_symmetry'),
    
    (pl.col('Tot Bwd Pkts') / (pl.col('Tot Fwd Pkts') + 1)).alias('response_ratio'),
    
    ((pl.col('Tot Fwd Pkts') > 0) & (pl.col('Tot Bwd Pkts') > 0)).cast(pl.Int32).alias('bidirectional_active')
])

print("Cluster 1: Traffic Balance (4 features added)")



Cluster 1: Traffic Balance (4 features added)


In [4]:
# ===== CELL 4: CLUSTER 2 - Temporal Patterns (3 features) =====
"""
PURPOSE: Identify abnormal timing patterns
- Automated attacks have very regular/irregular intervals
- Flooding attacks have extremely short intervals
- Normal traffic has natural variation
"""
df = df.with_columns([
    (pl.col('Flow IAT Std') / (pl.col('Flow IAT Mean') + 1)).alias('iat_regularity'),
    (total_pkts_expr / (pl.col('Flow Duration') + 0.001)).alias('flow_rate'),
    (pl.col('Fwd IAT Mean') - pl.col('Bwd IAT Mean')).abs().alias('timing_asymmetry')
])

print("Cluster 2: Temporal Patterns (3 features added)")



Cluster 2: Temporal Patterns (3 features added)


In [5]:
# ===== CELL 5: CLUSTER 3 - Packet Size Patterns (3 features) =====
"""
PURPOSE: Detect abnormal payload characteristics
- Many attacks use fixed-size packets (low variance)
- Scans often use minimal packets
- Exfiltration may show unusual size distributions
"""
df = df.with_columns([
    (pl.col('Pkt Len Std') / (pl.col('Pkt Len Mean') + 1)).alias('pkt_size_variance'),
    (pl.col('Fwd Pkt Len Mean') - pl.col('Bwd Pkt Len Mean')).abs().alias('fwd_bwd_size_diff'),
    (pl.col('Pkt Len Mean') < 100).cast(pl.Int32).alias('is_small_packet')
])

print("Cluster 3: Packet Size Patterns (3 features added)")



Cluster 3: Packet Size Patterns (3 features added)


In [6]:
# ===== CELL 6: CLUSTER 4 - Port Risk Indicators (3 features) =====
"""
PURPOSE: Flag potentially risky or unusual port usage
- Attacks often target specific vulnerable ports
- Ephemeral ports may indicate C&C communication
- Well-known ports have different risk profiles
"""
risky_ports = [21, 22, 23, 25, 445, 3389, 1433, 3306] 
web_ports = [80, 443, 8080, 8443]

df = df.with_columns([
    pl.col('Dst Port').is_in(risky_ports).cast(pl.Int32).alias('is_risky_port'),
    pl.col('Dst Port').is_in(web_ports).cast(pl.Int32).alias('is_web_traffic'),
    
    # pl.when().then() replaces pd.cut() for much faster binning
    pl.when(pl.col('Dst Port') <= 1024).then(0)
      .when(pl.col('Dst Port') <= 49151).then(1)
      .otherwise(2).alias('port_category')
])

print("Cluster 4: Port Risk Indicators (3 features added)")



Cluster 4: Port Risk Indicators (3 features added)


In [7]:
# ===== CELL 7: CLUSTER 5 - Composite Anomaly Score (1 feature) =====
"""
PURPOSE: Single metric combining multiple attack indicators
- High score = multiple suspicious characteristics present
- Useful for quick filtering and threshold-based detection
"""
df = df.with_columns([
    (
        (pl.col('traffic_symmetry') < 0.1).cast(pl.Int32) + 
        pl.col('is_small_packet') +  # Already Int32 from Cluster 3
        (pl.col('flow_rate') > pl.col('flow_rate').quantile(0.95)).cast(pl.Int32) + 
        (pl.col('iat_regularity') < 0.05).cast(pl.Int32) + 
        pl.col('is_risky_port')      # Already Int32 from Cluster 4
    ).alias('anomaly_score')
])

print("Cluster 5: Composite Anomaly Score (1 feature added)")



Cluster 5: Composite Anomaly Score (1 feature added)


In [11]:
# ===== CELL 8: Summary =====
print("\n" + "="*60)
print("FEATURE ENGINEERING COMPLETE")
print("="*60)
print(f"Final shape: {df.shape}")
print("\nAdded 14 high-impact features across 5 clusters")
print("\nNEW FEATURES:")
print("  - Traffic Balance (4): fwd_bwd_pkt_ratio, traffic_symmetry, response_ratio, bidirectional_active")
print("  - Temporal (3): iat_regularity, flow_rate, timing_asymmetry")
print("  - Packet Size (3): pkt_size_variance, fwd_bwd_size_diff, is_small_packet")
print("  - Port Risk (3): is_risky_port, is_web_traffic, port_category")
print("  - Anomaly Score (1): anomaly_score")

# Display first few rows of new features
print("\nSample of new features:")
new_cols = ['fwd_bwd_pkt_ratio', 'traffic_symmetry', 'response_ratio', 'bidirectional_active',
            'iat_regularity', 'flow_rate','timing_asymmetry','pkt_size_variance','fwd_bwd_size_diff','is_web_traffic','port_category', 'is_small_packet', 'is_risky_port', 'anomaly_score']

# Using pl.Config to ensure all columns print clearly
with pl.Config(tbl_cols=-1):
    print(df.select(new_cols).head())



FEATURE ENGINEERING COMPLETE
Final shape: (5529767, 43)

Added 14 high-impact features across 5 clusters

NEW FEATURES:
  - Traffic Balance (4): fwd_bwd_pkt_ratio, traffic_symmetry, response_ratio, bidirectional_active
  - Temporal (3): iat_regularity, flow_rate, timing_asymmetry
  - Packet Size (3): pkt_size_variance, fwd_bwd_size_diff, is_small_packet
  - Port Risk (3): is_risky_port, is_web_traffic, port_category
  - Anomaly Score (1): anomaly_score

Sample of new features:
shape: (5, 14)
┌─────┬─────┬─────┬─────┬─────┬─────┬─────┬────────┬───────┬───────┬───────┬───────┬───────┬───────┐
│ fwd ┆ tra ┆ res ┆ bid ┆ iat ┆ flo ┆ tim ┆ pkt_si ┆ fwd_b ┆ is_we ┆ port_ ┆ is_sm ┆ is_ri ┆ anoma │
│ _bw ┆ ffi ┆ pon ┆ ire ┆ _re ┆ w_r ┆ ing ┆ ze_var ┆ wd_si ┆ b_tra ┆ categ ┆ all_p ┆ sky_p ┆ ly_sc │
│ d_p ┆ c_s ┆ se_ ┆ cti ┆ gul ┆ ate ┆ _as ┆ iance  ┆ ze_di ┆ ffic  ┆ ory   ┆ acket ┆ ort   ┆ ore   │
│ kt_ ┆ ymm ┆ rat ┆ ona ┆ ari ┆ --- ┆ ymm ┆ ---    ┆ ff    ┆ ---   ┆ ---   ┆ ---   ┆ ---   ┆ ---  

In [12]:


# ===== CELL 9 (Optional): Save Dataset =====
df.write_parquet("train_feature_engineered.parquet")
print("Dataset saved!")

Dataset saved!


In [15]:
# load feature engineered dataset to check new features
df = pl.read_parquet("train_feature_engineered.parquet")
#print column names to verify new features are present
df.collect_schema().names()


['Timestamp',
 'Dst Port',
 'Protocol',
 'Flow Duration',
 'Tot Fwd Pkts',
 'Tot Bwd Pkts',
 'Flow Byts/s',
 'Flow Pkts/s',
 'Fwd Pkts/s',
 'Bwd Pkts/s',
 'Pkt Len Mean',
 'Pkt Len Var',
 'Pkt Len Std',
 'Fwd Pkt Len Mean',
 'Bwd Pkt Len Mean',
 'Fwd Pkt Len Std',
 'Bwd Pkt Len Std',
 'Fwd Seg Size Avg',
 'Bwd Seg Size Avg',
 'Init Fwd Win Byts',
 'Init Bwd Win Byts',
 'Active Mean',
 'Idle Mean',
 'Flow IAT Mean',
 'Flow IAT Std',
 'Fwd IAT Mean',
 'Bwd IAT Mean',
 'Down/Up Ratio',
 'Label',
 'fwd_bwd_pkt_ratio',
 'traffic_symmetry',
 'response_ratio',
 'bidirectional_active',
 'iat_regularity',
 'flow_rate',
 'timing_asymmetry',
 'pkt_size_variance',
 'fwd_bwd_size_diff',
 'is_small_packet',
 'is_risky_port',
 'is_web_traffic',
 'port_category',
 'anomaly_score']

In [16]:
len(df.collect_schema().names())


43